# Financial RAG System
PDFs → Load → Chunk → Embed → CHROMA → Retrieve → Answer



In [1]:
from pathlib import Path
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

print("Imports done")

C:\Users\Bipasha Ray\AppData\Local\Temp\ipykernel_16460\4131419562.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports done


In [2]:
DATA_DIR        = Path("../financial-rag-agent/data/raw")
VECTORSTORE_DIR = Path("../vectorstore/chroma")
LLM_MODEL       = "llama3.2"
EMBED_MODEL     = "nomic-embed-text"
COLLECTION      = "financial_docs"
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 100
TOP_K           = 5

DATA_DIR.mkdir(parents=True, exist_ok=True)
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDFs   : {DATA_DIR.resolve()}")
print(f"ChromaDB : {VECTORSTORE_DIR.resolve()}")

PDFs   : C:\Dev\financial-rag-agent\financial-rag-agent\data\raw
ChromaDB : C:\Dev\financial-rag-agent\vectorstore\chroma


In [3]:
#Load documents 

pdf_files = sorted(DATA_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs")

docs = []
for pdf in tqdm(pdf_files, desc="Loading"):
    pages = PyPDFLoader(str(pdf)).load()
    for page in pages:
        page.metadata["source"] = pdf.name
    docs.extend(pages)

print(f"Total pages loaded: {len(docs)}")

Found 22 PDFs


Loading: 100%|██████████| 22/22 [07:14<00:00, 19.74s/it]

Total pages loaded: 4091


In [4]:
# Verify one document
print("Source  :", docs[0].metadata["source"])
print("Page    :", docs[0].metadata.get("page"))
print("Content :", docs[0].page_content[:300])

Source  : Annual Economic Report 2026.pdf
Page    : 0
Content : Annual EconomicAnnual Economic
Report
June 2026


In [5]:
#Chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(docs)
avg    = sum(len(c.page_content) for c in chunks) // len(chunks)

print(f"Pages   : {len(docs)}")
print(f"Chunks  : {len(chunks)}")
print(f"Avg size: {avg} chars")

Pages   : 4091
Chunks  : 31035
Avg size: 438 chars


In [6]:
# Verify one chunk
print("Source  :", chunks[10].metadata["source"])
print("Page    :", chunks[10].metadata.get("page"))
print("Content :", chunks[10].page_content)

Source  : Annual Economic Report 2026.pdf
Page    : 4
Content : Policy implications ...........................................................................................................................................  28 
Endnotes .............................................................................................................................................................  33 
Additional notes to graphs ...........................................................................................................................  33


In [7]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

test = embeddings.embed_query(" What is inflation ")
print(f"Vector size  : {len(test)}")
print(f"First 5 vals : {test[:5]}")

Vector size  : 768
First 5 vals : [0.042089183, 0.08365569, -0.1812736, -0.022868775, 0.00026002576]


In [8]:
# Build ChromaDB
BATCH = 100

vectorstore = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

for i in tqdm(range(0, len(chunks), BATCH), desc="Embedding"):
    vectorstore.add_documents(chunks[i : i + BATCH])

print(f"Indexed {len(chunks)} chunks")
print(f"Saved  : {VECTORSTORE_DIR.resolve()}")

Embedding: 100%|██████████| 311/311 [4:39:52<00:00, 54.00s/it]      

Indexed 31035 chunks
Saved  : C:\Dev\financial-rag-agent\vectorstore\chroma


In [9]:
# Load from disk and verify
db = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

print(f"Total chunks in DB: {db._collection.count()}")

Total chunks in DB: 31035


In [10]:
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": TOP_K, "fetch_k": 20, "lambda_mult": 0.7},
)

# Test retrieval
results = retriever.invoke("What is the inflation rate forecast?")

for i, doc in enumerate(results, 1):
    print(f"\n[{i}] {doc.metadata.get('source')} | page {doc.metadata.get('page')}")
    print(doc.page_content[:200])


[1] INDIAN ECONOMIC SURVEY 2025-2026.pdf | page 256
-2
0
2
4
6
8
10
-3 - 1 13579
Inflation Rate (per cent)
GDP Growth Rate (per cent)
Source: Drawn using the data from the World Economic Outlook Database, IMF

[2] BIS Annual Economic Report 2025.pdf | page 31
mark on inflation expectations. 
Survey evidence across a sample of 29 advanced and emerging market economies shows that, on average,  
households expect inflation over the next 12 months to be about 

[3] Economic Bulletin.pdf | page 2
rate decisions will be based on its assessment of the inflation outlook and the risks 
surrounding it, in light of the incoming economic and financial data, as well as the 
dynamics of underlying infl

[4] OECD Economic Outlook, Volume 2025.pdf | page 279
inflation expectations, and survey forecasts. This model decomposes observed interest rates into expected inflation, term pre miums, and the 
underlying real rate of interest. 
Source: Bureau of Econo

[5] Annual Report 23-24.pdf | page 109
revea

In [11]:
PROMPT = ChatPromptTemplate.from_template("""
You are a financial analyst assistant.
Answer using ONLY the context below.
If the answer is not in the context, say "I don't have that information."
Cite source using [filename, page] format.

Context:
{context}

Question: {question}

Answer:
""")

print("Prompt ready")

Prompt ready


In [ ]:
llm = ChatOllama(model=LLM_MODEL, temperature=0)

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source','unknown')}, page {d.metadata.get('page','?')}]\n{d.page_content}"
        for d in docs
    )

def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return {
        "context":  format_docs(docs),
        "question": question,
        "sources":  docs,
    }

chain = (
    RunnableLambda(retrieve_and_format)
    | RunnablePassthrough.assign(answer=PROMPT | llm | StrOutputParser())
)

print("Chain ready")

In [ ]:
questions = [
    "What is the inflation rate forecast?",
    "What does the report say about gold demand?",
    "What are the key risk factors mentioned?",
]

for q in questions:
    result  = chain.invoke(q)
    sources = sorted({d.metadata.get("source") for d in result["sources"]})
    print(f"Q : {q}")
    print(f"A : {result['answer']}")
    print(f"Sources : {', '.join(sources)}")
    print("=" * 55)